In [5]:
%pip install pandas openpyxl --q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Consolidar Rankings em um único Excel

Lê todos os arquivos `Ranking_*.xlsx` de uma pasta e grava tudo em **um único arquivo
Excel**, onde cada arquivo original vira uma **aba**.

**Como usar**: rode a célula de configuração abaixo, ajuste `PASTA_ENTRADA` se os
arquivos não estiverem na mesma pasta deste notebook, depois rode as células
seguintes em ordem.

### 1. Configuração

In [6]:
import re
import glob
import os
import pandas as pd

PASTA_ENTRADA = "."  # pasta onde estao os Ranking_*.xlsx ("." = mesma pasta deste notebook)
ARQUIVO_SAIDA = os.path.join(PASTA_ENTRADA, "Rankings_Consolidado.xlsx")

# Caracteres que o Excel nao aceita em nome de aba, e o limite de 31 caracteres.
CARACTERES_INVALIDOS = re.compile(r'[\[\]:*?/\\]')
LIMITE_NOME_ABA = 31

print(f"Pasta de entrada: {os.path.abspath(PASTA_ENTRADA)}")
print(f"Arquivo de saida: {os.path.abspath(ARQUIVO_SAIDA)}")


Pasta de entrada: c:\Users\USUARIO\Desktop\doutorado\Códigos para o Artigo
Arquivo de saida: c:\Users\USUARIO\Desktop\doutorado\Códigos para o Artigo\Rankings_Consolidado.xlsx


### 2. Funções

In [7]:
def nome_aba_valido(nome, nomes_ja_usados):
    """Limpa o nome do arquivo para virar um nome de aba valido no Excel (sem
    caracteres proibidos, ate 31 caracteres, sem repetir um nome ja usado)."""
    nome = CARACTERES_INVALIDOS.sub("_", nome)
    nome = nome[:LIMITE_NOME_ABA]

    candidato = nome
    sufixo = 1
    while candidato in nomes_ja_usados:
        sufixo += 1
        corte = LIMITE_NOME_ABA - len(f"_{sufixo}")
        candidato = f"{nome[:corte]}_{sufixo}"
    return candidato


def consolidar(pasta_entrada: str, caminho_saida: str) -> None:
    padrao = os.path.join(pasta_entrada, "Ranking_*.xlsx")
    arquivos = sorted(glob.glob(padrao))

    if not arquivos:
        print(f"Nenhum arquivo \'Ranking_*.xlsx\' encontrado em: {pasta_entrada}")
        return

    print(f"{len(arquivos)} arquivo(s) encontrado(s) em: {pasta_entrada}")

    nomes_usados = set()
    abas_gravadas = 0

    with pd.ExcelWriter(caminho_saida, engine="openpyxl") as writer:
        for caminho in arquivos:
            nome_arquivo = os.path.splitext(os.path.basename(caminho))[0]
            # "Ranking_Cp_Original.xlsx" -> aba "Cp_Original"
            nome_base = nome_arquivo[len("Ranking_"):] if nome_arquivo.startswith("Ranking_") else nome_arquivo

            try:
                df = pd.read_excel(caminho)
            except Exception as e:
                print(f"  AVISO: erro ao ler \'{os.path.basename(caminho)}\': {e} -- pulando.")
                continue

            nome_aba = nome_aba_valido(nome_base, nomes_usados)
            nomes_usados.add(nome_aba)

            df.to_excel(writer, sheet_name=nome_aba, index=False)
            abas_gravadas += 1
            marca = f" (aba: \'{nome_aba}\')" if nome_aba != nome_base else ""
            print(f"  OK: {os.path.basename(caminho)}{marca}")

    print(f"\nConcluido: {abas_gravadas} aba(s) gravada(s) em: {caminho_saida}")


### 3. Execução

In [8]:
consolidar(PASTA_ENTRADA, ARQUIVO_SAIDA)


43 arquivo(s) encontrado(s) em: .


  OK: Ranking_ANOVA.xlsx
  OK: Ranking_AmplitudeSigma.xlsx
  OK: Ranking_AmplitudeSigma_BaseCompleta.xlsx
  OK: Ranking_ApproxMIC.xlsx
  OK: Ranking_Assimetria.xlsx
  OK: Ranking_Assimetria_BaseCompleta.xlsx
  OK: Ranking_Chi2.xlsx
  OK: Ranking_Cp_Normalizado.xlsx
  OK: Ranking_Cp_Original.xlsx
  OK: Ranking_Cpk_Normalizado.xlsx
  OK: Ranking_Cpk_Original.xlsx
  OK: Ranking_Curtose.xlsx
  OK: Ranking_Curtose_BaseCompleta.xlsx
  OK: Ranking_FScore_ANOVA.xlsx
  OK: Ranking_FisherScore.xlsx
  OK: Ranking_GainRatio.xlsx
  OK: Ranking_GiniIndex.xlsx
  OK: Ranking_HellingerDistance.xlsx
  OK: Ranking_IQRSigma.xlsx
  OK: Ranking_IQRSigma_BaseCompleta.xlsx
  OK: Ranking_InformationGain.xlsx
  OK: Ranking_KS.xlsx
  OK: Ranking_Kendall.xlsx
  OK: Ranking_LaplacianScore.xlsx
  OK: Ranking_Pearson.xlsx
  OK: Ranking_Pp_Normalizado.xlsx
  OK: Ranking_Pp_Original.xlsx
  OK: Ranking_Ppk_Normalizado.xlsx
  OK: Ranking_Ppk_Original.xlsx
  OK: Ranking_ReliefF.xlsx
  OK: Ranking_SPEC.xlsx
  OK: Ranking_